# Mélo et la commande optimale

Ce notebook explique la résolution d'un problème de commande optimale : 
Mélo doit atteindre une pomme en évitant un hérisson.

Nous allons :
1. Définir les entités du problème
2. Construire l’environnement
3. Résoudre l’OCP
4. Visualiser la trajectoire

On commence par importer les librairies dont on a besoin : 

In [5]:
import sys
import os

sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../src"))

import numpy as np
from gekko import GEKKO

from Environnement import Environnement 
from Melo import Melo
from Herisson import Herisson
from Entite import Entite
from affichage import visualiser_simulation

Création du modèle d'optimisation

In [3]:
m = GEKKO(remote=False)

On créer ensuite les différentes entités. 
On commence par Mélo : 
`x0_melo` est le vecteur qui contient la position et la vitesse initiale de mélo. 
`image_melo` et `taille_melo` servent uniquement pour l'animation. 
On déclare ensuite un objet `melo` (le système). 

In [6]:
# Mélo
x0_melo = np.concatenate((np.array([1.5, 7]), np.zeros(2)))
image_melo = "src/melo.png"
taille_melo = 2.5
melo = Melo(m, "Mélo", x0_melo, image_melo, taille_melo)

FileNotFoundError: [Errno 2] No such file or directory: 'src/melo.png'

On créer ensuite la pomme et le pommier. 
Encore une, `image` et `taille` servent uniquement pour l'animation. 
`x_pomme` est le vecteur position de la pomme. 
C'est aussi la position finale que cherche à atteindre Mélo. 

In [ ]:
# Pomme
x_pomme = np.array([4.5, 2.5]) 
image_pomme = "src/pomme.png"
taille_pomme = 1.25
pomme = Entite("Pomme", x_pomme, image_pomme, taille_pomme)

# Pommier (pour affichage)
x_pommier = np.array([5.8, 3.8]) 
image_pommier = "src/pommier.png"
taille_pommier = 4
pommier = Entite("", x_pommier, image_pommier, taille_pommier)

Enfin, on créer le hérisson. 
Ceci définit un obstacle à éviter. Il est modélisé par un cercle de rayon `rayon_herisson` et de centre `x_herisson`. 

On stocke ensuite ce hérisson dans une liste appellé `obstacles`. 

In [ ]:
# Hérisson
x_herisson = np.array([3,4.5])
rayon_herisson = 1
image_herisson = "src/herisson.png"
taille_herisson = 2
herisson = Herisson("Hérisson", x_herisson, rayon_herisson, image_herisson, taille_herisson)

obstacles = [herisson]
entites = [pomme, pommier, herisson]

Finalement, on créer l'environnement contenant tous les éléments du problème. 
On définit d'abord le temps total de la simulation ainsi que le nombre de pas. 
Dit autrement, on découpe le temps total en `N` petit bout régulier. 
On créer ensuite l'environnement. 

In [ ]:
temps_total = 5 # s
N = 100 # nombre de pas de temps 

env = Environnement(m, temps_total, N, melo, pomme, obstacles, entites)

Il n'y a plus qu'à setup le problème de commande optimale et le résoudre avec `solve()`. 
On affiche ensuite les résultats. 

In [ ]:
env.setup_ocp()
env.solve()

# Résultats
print("Position finale de Melo :", env.melo.x.VALUE[-1], env.melo.y.VALUE[-1])
print("Vitesse finale de Melo  :", env.melo.vx.VALUE[-1], env.melo.vy.VALUE[-1])

Et pour finir, on créer une animation de la résolution. 

In [ ]:
chemin_sauvegarde = "src/simulation.gif"
visualiser_simulation(env, chemin_sauvegarde)